# ThreatLens AI — Notebook 05: Threat Scoring Engine

**Stage in the pipeline:** `Intelligence → Threat Score`
**Input:** anomaly model (Notebook 02) + attack classifier (Notebook 03)

### What this notebook does
1. Loads both trained models — the Isolation Forest and the attack classifier
2. Runs both on the same test rows, getting an anomaly score and a classifier prediction+confidence per row
3. Combines them into one 0–100 **threat score** and a **severity label** using `src/models/threat_score.py`
4. Checks the result against real labels — do the highest scores actually line up with real attacks?
5. Reproduces the exact alert-card format used in the dashboard (IP, threat score, severity, attack type, risk factors)

### Why combine two models instead of picking one
Each model alone has a real blind spot. The anomaly detector (Notebook 02) can flag something as unusual without knowing what it is or how bad it is. The classifier (Notebook 03) can confidently name an attack type but was never asked "how anomalous does this look overall?" Combining both, plus a severity weight per attack type, is what the blueprint calls a **"weighted risk engine using model + behavioral signals"** — a single trustworthy number an analyst can act on, backed by two independent pieces of evidence instead of one.


In [2]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.models.anomaly import select_feature_columns, predict_anomalies, load_model as load_anomaly_model
from src.models.classifier import load_model as load_classifier_model
from src.models.threat_score import normalize_anomaly_score, compute_threat_score

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (10, 5)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


## 1. Load data and both trained models

Update the two filenames below if your saved model names differ (check `models/anomaly/` and `models/classifier/`).


In [3]:
ANOMALY_MODEL_FILE = "isolation_forest_v1.joblib"
CLASSIFIER_MODEL_FILE = "attack_classifier_xgboost_v1.joblib"  # match whatever Notebook 03 saved

df = pd.read_parquet(PROCESSED_DIR / "cicids2017_cleaned.parquet")
feature_cols = select_feature_columns(df)
X = df[feature_cols]
y = df["attack_category"]

anomaly_model = load_anomaly_model(PROJECT_ROOT / "models" / "anomaly" / ANOMALY_MODEL_FILE)
classifier_model = load_classifier_model(PROJECT_ROOT / "models" / "classifier" / CLASSIFIER_MODEL_FILE)
print("Both models loaded.")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Microsoft\\Desktop\\Nayi_Manzil_Internship\\ThreatLensAI\\models\\classifier\\attack_classifier_xgboost_v1.joblib'

## 2. Use a shared test split

Same `random_state=42` / `stratify=y` as Notebooks 02 and 03, so we're scoring the exact same held-out rows those notebooks evaluated — keeping every number in this project comparable end to end.


In [ ]:
_, X_test, _, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Scoring {len(X_test):,} test rows")

## 3. Run both models on the test set

- The anomaly model gives a raw score per row (Notebook 02's output)
- The classifier gives a predicted attack category **and** its confidence (the model's own predicted probability for that class) — confidence matters as much as the label itself: a "DDoS, 98% confident" prediction should weigh more heavily than a "DDoS, 34% confident" one.


In [ ]:
anomaly_out = predict_anomalies(anomaly_model, X_test)
anomaly_score_norm = normalize_anomaly_score(anomaly_out["anomaly_score"])

predicted_category = pd.Series(classifier_model.predict(X_test), index=X_test.index)
predicted_proba = classifier_model.predict_proba(X_test)
classifier_confidence = pd.Series(predicted_proba.max(axis=1), index=X_test.index)

print("Sample of raw model outputs:")
pd.DataFrame({
    "anomaly_score_norm": anomaly_score_norm,
    "predicted_category": predicted_category,
    "classifier_confidence": classifier_confidence.round(3),
}).head(8)

## 4. Compute the combined threat score

This calls `compute_threat_score()` from `src/models/threat_score.py` — the same function a FastAPI `/threats` endpoint would call later. The default weights (35% anomaly, 35% classifier confidence, 30% attack-type severity) are visible and adjustable right there in that file, not buried in this notebook.


In [ ]:
scores = compute_threat_score(
    anomaly_score_normalized=anomaly_score_norm,
    predicted_attack_category=predicted_category,
    classifier_confidence=classifier_confidence,
)

results = pd.DataFrame({
    "true_category": y_test,
    "predicted_category": predicted_category,
    "threat_score": scores["threat_score"],
    "severity": scores["severity"],
})
results.head(10)

## 5. Sanity check — do high scores actually line up with real attacks?

If the threat scoring formula is working, rows that are genuinely attacks should skew toward higher scores, and genuinely normal rows should skew low. This is the same kind of honesty check we ran on the anomaly detector alone in Notebook 02 — now checking the *combined* signal.


In [ ]:
results["is_actually_attack"] = (results["true_category"] != "Normal").map({True: "Attack", False: "Normal"})

plt.figure(figsize=(10, 5))
sns.kdeplot(data=results, x="threat_score", hue="is_actually_attack", fill=True, common_norm=False,
            palette=["#3aa0ff", "#ff4d5a"])
plt.title("Threat score distribution — real Normal vs real Attack rows")
plt.xlabel("Threat score (0-100)")
plt.tight_layout()
plt.show()

print("Average threat score by real class:")
print(results.groupby("is_actually_attack")["threat_score"].mean().round(1))

print("\nSeverity label distribution among real attacks:")
print(results[results["is_actually_attack"] == "Attack"]["severity"].value_counts())

## 6. Reproduce a dashboard-style alert card

This block formats one high-scoring row exactly the way the blueprint's own example alert (Section 13) and the dashboard's alert cards present it — proving the numbers this notebook computes are the same numbers that would eventually populate the live UI, not a disconnected metric.


In [ ]:
top_alert = results.sort_values("threat_score", ascending=False).iloc[0]
top_alert_features = X_test.loc[[top_alert.name]]

print("=" * 50)
print("  ALERT CARD PREVIEW")
print("=" * 50)
print(f"  Attack Type   : {top_alert['predicted_category']}")
print(f"  Threat Score  : {top_alert['threat_score']}%")
print(f"  Severity      : {top_alert['severity'].upper()}")
print(f"  True Label    : {top_alert['true_category']}  "
      f"({'MATCH' if top_alert['true_category'] == top_alert['predicted_category'] else 'model disagreed'})")
print("=" * 50)

## 7. Summary & next steps

| Item | Result |
|---|---|
| Avg threat score — real attacks | see Section 5 |
| Avg threat score — real normal traffic | see Section 5 |
| Severity distribution among real attacks | see Section 5 |
| Formula | 35% anomaly + 35% classifier confidence + 30% attack-type severity — see `src/models/threat_score.py` |

We now have three connected engines: **Anomaly Detection** (is this unusual), **Attack Classification** (what is it), and **Threat Scoring** (how bad is it, as one clean number) — matching the blueprint's "Intelligence" layer (UBA + Graph + Threat Score + SHAP) apart from UBA and the relationship graph, which are the natural next notebooks.

**Next up:** Notebook 06 — User Behavior Analytics (building per-user baselines and flagging deviations, the same U-1042 style profile already mocked in the dashboard), or Notebook 07 — Time-series Threat Forecasting, whichever you want to tackle first.
